In [7]:
import pandas as pd
from pathlib import Path

def cruzar_estado(excel_path, json_path, output_path):
    # 1) Leo el Excel y normalizo sólo los dígitos en _proj_int
    df_xl = pd.read_excel(excel_path, dtype=str)
    df_xl['_proj_int'] = (
        df_xl['NumeroProyecto']
           .str.extract(r'(\d+)', expand=False)  # toma sólo números
           .fillna('0')                          # convierte NaN → '0'
           .astype(int)
    )

    # 2) Cargo el JSON y construyo el mapa
    df_j = pd.read_json(json_path, orient='records', dtype=str)
    registros = []

    for row in df_j.to_dict(orient='records'):
        # Archivados
        pa = row.get("Proyectos Archivados")
        aa = row.get("Año Archivados")
        if isinstance(pa, str) and pa.isdigit():
            num = int(pa)
            # sólo añado si el año también es un dígito válido
            year = int(aa) if isinstance(aa, str) and aa.isdigit() else None
            registros.append({
                '_proj_int': num,
                'estado': 'archivado',
                'anio_estado': year
            })

        # Retirados
        pr = row.get("Proyectos Retirados")
        ar = row.get("Año Retirados")
        if isinstance(pr, str) and pr.isdigit():
            num = int(pr)
            year = int(ar) if isinstance(ar, str) and ar.isdigit() else None
            registros.append({
                '_proj_int': num,
                'estado': 'retirado',
                'anio_estado': year
            })

    df_map = pd.DataFrame(registros)
    # Si un proyecto sale en ambas categorías, nos quedamos con 'retirado'
    df_map = df_map.drop_duplicates(subset='_proj_int', keep='last')

    # 3) Merge con el Excel
    df_out = df_xl.merge(df_map, on='_proj_int', how='left')

    # 4) Limpio columna auxiliar y guardo
    df_out = df_out.drop(columns=['_proj_int'])
    df_out.to_excel(output_path, index=False)
    print(f"✅ Generado: {Path(output_path).resolve()}")

if __name__ == '__main__':
    cruzar_estado(
        excel_path  = '2018-2019.xlsx',           # tu Excel
        json_path   = 'novedades_v1.json',        # tu JSON de proyectos
        output_path = '2018-2019_con_estado.xlsx' # salida deseada
    )


✅ Generado: C:\Users\juans\Documents\proarchitecg\Model-Extract-information\extract\modelos\team\2018-2019_con_estado.xlsx


In [3]:
import pandas as pd
import unicodedata

def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    cols = []
    for c in df.columns:
        # 1) quitar acentos
        nc = "".join(ch for ch in unicodedata.normalize("NFD", str(c))
                     if unicodedata.category(ch) != "Mn")
        # 2) a minúsculas, sin espacios, con _  
        nc = nc.strip().lower().replace(" ", "_")
        cols.append(nc)
    df.columns = cols
    return df

# 1) Carga tus archivos
df_base   = pd.read_excel("2022_2023.xlsx")
df_fichas = pd.read_excel("fichas_tecnicas_con_estado_2022_2023.xlsx")

# 2) Normaliza columnas
df_base   = clean_column_names(df_base)
df_fichas = clean_column_names(df_fichas)

# 3) Inspecciona para ver cómo quedaron
print("Columnas df_base  :", df_base.columns.tolist())
print("Columnas df_fichas:", df_fichas.columns.tolist())

# --- Ahora sustituye estos nombres por los que realmente veas ahí ---
# Por ejemplo, supongamos que tras limpiar tienes:
#   df_base    → 'numeroproyecto'
#   df_fichas  → 'num_camara', 'num_senado', 'estado_actual'

# 4) Construye los mapas de búsqueda
map_estado_camara = df_fichas.set_index("num_camara")["estado_actual"].to_dict()
map_estado_sena   = df_fichas.set_index("num_senado")["estado_actual"].to_dict()

def lookup_estado(proy):
    # Si coincide con num_camara
    if proy in map_estado_camara:
        return proy, None, map_estado_camara[proy]
    # Si coincide con num_senado
    if proy in map_estado_sena:
        return None, proy, map_estado_sena[proy]
    return None, None, "sin novedad"

# 5) Aplica a la columna de proyectos
df_base[["num_camara_match","num_senado_match","estado_actual"]] = \
    df_base["numeroproyecto"].apply(lookup_estado).tolist()

# 6) Guarda el resultado
df_base.to_excel("2022_2023_con_estado.xlsx", index=False)
print("✅ Hecho: 2022_2023_con_estado.xlsx")



Columnas df_base  : ['name', 'asunto', 'titulounidaddoc', 'cronologicos', 'fecha_inicial', 'fecha_final', 'legislatura', 'numeroproyecto']
Columnas df_fichas: ['num_camara', 'num_senado', 'fecha_radicacion', 'tipo_proyecto', 'seudonimo', 'comision', 'camara_origen', 'titulo', 'autores', 'estado_actual']
✅ Hecho: 2022_2023_con_estado.xlsx
